# DATATHON 2026 - EDA Notebook

Structure:
1. Imports & Config
2. Data Quality & Cleaning
3. Descriptive Analysis
4. Diagnostic Analysis
5. Predictive Analysis
6. Prescriptive Analysis

## 1 - imports và config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')


DATA_DIR = 'Data/'

# Load toàn bộ data
sales = pd.read_csv(DATA_DIR + 'sales.csv',        parse_dates=['Date'])
orders = pd.read_csv(DATA_DIR + 'orders.csv',       parse_dates=['order_date'])
items = pd.read_csv(DATA_DIR + 'order_items.csv')
products = pd.read_csv(DATA_DIR + 'products.csv')
customers = pd.read_csv(DATA_DIR + 'customers.csv',    parse_dates=['signup_date'])
geo = pd.read_csv(DATA_DIR + 'geography.csv')
promos = pd.read_csv(DATA_DIR + 'promotions.csv',   parse_dates=['start_date','end_date'])
payments = pd.read_csv(DATA_DIR + 'payments.csv')
ships = pd.read_csv(DATA_DIR + 'shipments.csv',    parse_dates=['ship_date','delivery_date'])
returns = pd.read_csv(DATA_DIR + 'returns.csv',      parse_dates=['return_date'])
reviews = pd.read_csv(DATA_DIR + 'reviews.csv')
inventory = pd.read_csv(DATA_DIR + 'inventory.csv',    parse_dates=['snapshot_date'])
traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv',  parse_dates=['date'])

print('Loaded all tables')
for name, df in [('sales',sales),('orders',orders),('items',items),
                 ('products',products),('customers',customers),
                 ('returns',returns),('inventory',inventory),('traffic',traffic)]:
    print(f'  {name:12s}: {df.shape}')

## 2 - data quality và clean

In [ ]:
#1null audit 
tables = {
    'sales': sales, 'orders': orders, 'items': items,
    'products': products, 'customers': customers,
    'returns': returns, 'traffic': traffic
}
print('null audit')
for name, df in tables.items():
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0]
    if len(null_cols):
        print(f'\n{name}:')
        for col, n in null_cols.items():
            print(f'  {col}: {n} nulls ({n/len(df)*100:.1f}%)')
    else:
        print(f'{name}: no nulls')

In [ ]:
#2 duplicate check
print('duplicate check')
for name, df in tables.items():
    dupes = df.duplicated().sum()
    flag = 'error' if dupes > 0 else 'done'
    print(f'{name:12s}: {dupes} duplicate rows {flag}')

In [ ]:
#3business constraint check
print('business constraint check')

#cogs < price for every product
bad_margin = products[products['cogs'] >= products['price']]
print(f'products với cogs >= price: {len(bad_margin)} rows {"error" if len(bad_margin) else "done"}')

#revenue > 0
bad_rev = sales[sales['Revenue'] <= 0]
print(f'sales với Revenue <= 0: {len(bad_rev)} rows {"error" if len(bad_rev) else "done"}')

# COGS > 0
bad_cogs = sales[sales['COGS'] <= 0]
print(f'sales với COGS <= 0: {len(bad_cogs)} rows {"error" if len(bad_cogs) else "done"}')

#return_quantity > 0
bad_ret = returns[returns['return_quantity'] <= 0]
print(f'returns với quantity <= 0: {len(bad_ret)} rows {"error" if len(bad_ret) else "done"}')

#date range
print(f'\nsales date range: {sales.Date.min().date()} => {sales.Date.max().date()}')

In [ ]:
# 4 outlier check trên sale
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['Revenue', 'COGS']):
    ax.boxplot(sales[col].dropna(), vert=False)
    ax.set_title(f'Outlier check: {col}')
    ax.set_xlabel('VND')
plt.suptitle('Boxplot Revenue & COGS - kiểm tra outlier', fontsize=13)
plt.tight_layout()
plt.savefig('chart_dq_outliers.png', dpi=150)
plt.show()

for col in ['Revenue', 'COGS']:
    q1, q3 = sales[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    out = sales[(sales[col] < q1 - 1.5*iqr) | (sales[col] > q3 + 1.5*iqr)]
    print(f'{col}: {len(out)} outlier rows ({len(out)/len(sales)*100:.1f}%)')

## 3 - descriptive analysis

In [ ]:
# chart 1: Revenue vs COGS theo tháng
monthly = sales.resample('M', on='Date').sum().reset_index()
monthly['Profit_Margin'] = (monthly['Revenue'] - monthly['COGS']) / monthly['Revenue'] * 100
monthly['year'] = monthly['Date'].dt.year
monthly['month'] = monthly['Date'].dt.month

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['Date'], monthly['Revenue'], label='Revenue', linewidth=2)
ax.plot(monthly['Date'], monthly['COGS'],    label='COGS',    linewidth=2, linestyle='--')
ax.fill_between(monthly['Date'], monthly['COGS'], monthly['Revenue'],
                alpha=0.15, label='Gross Profit')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))
ax.set_title('Revenue vs COGS theo tháng (2012-2022)', fontsize=14)
ax.set_xlabel('Tháng'); ax.set_ylabel('VND (triệu)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('chart1_revenue_cogs.png', dpi=150)
plt.show()

print(f'Tổng Revenue (train): {sales.Revenue.sum():,.0f} VND')
print(f'Tổng COGS    (train): {sales.COGS.sum():,.0f} VND')
print(f'Gross Profit margin TB: {monthly.Profit_Margin.mean():.1f}%')

In [ ]:
# chart 2: Profit Margin % theo tháng
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly['Date'], monthly['Profit_Margin'], color='steelblue', linewidth=2)
mean_pm = monthly['Profit_Margin'].mean()
ax.axhline(mean_pm, color='red', linestyle='--', alpha=0.5,
           label=f'Trung bình: {mean_pm:.1f}%')
ax.set_title('Profit Margin (%) theo tháng (2012-2022)', fontsize=14)
ax.set_xlabel('Tháng'); ax.set_ylabel('Profit Margin (%)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('chart2_profit_margin.png', dpi=150)
plt.show()

print('Tháng margin thấp nhất:')
print(monthly.nsmallest(3, 'Profit_Margin')[['Date','Profit_Margin']].to_string())
print('\nTháng margin cao nhất:')
print(monthly.nlargest(3, 'Profit_Margin')[['Date','Profit_Margin']].to_string())

In [ ]:
# chart 3: Heatmap Profit Margin năm x tháng
pivot = monthly.pivot(index='year', columns='month', values='Profit_Margin')
pivot.columns = ['T'+str(c) for c in pivot.columns]

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Profit Margin (%)'})
ax.set_title('Profit Margin (%) theo Năm x Tháng', fontsize=14)
ax.set_xlabel('Tháng'); ax.set_ylabel('Năm')
plt.tight_layout()
plt.savefig('chart3_heatmap.png', dpi=150)
plt.show()

In [ ]:
# chart 4: Doanh số theo Category & Segment
items_prod = items.merge(products[['product_id','category','segment','price','cogs']], on='product_id')
items_prod['revenue_line'] = items_prod['quantity'] * items_prod['unit_price'] - items_prod['discount_amount']
items_prod['cogs_line'] = items_prod['quantity'] * items_prod['cogs']

cat_rev = items_prod.groupby('category')['revenue_line'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cat_rev.index, cat_rev.values / 1e6, color='steelblue')
axes[0].set_title('Doanh thu theo Category (triệu VND)', fontsize=13)
axes[0].set_xlabel('Category'); axes[0].set_ylabel('Triệu VND')
axes[0].tick_params(axis='x', rotation=30)

seg_rev = items_prod.groupby('segment')['revenue_line'].sum().sort_values(ascending=False)
axes[1].bar(seg_rev.index, seg_rev.values / 1e6, color='tomato')
axes[1].set_title('Doanh thu theo Segment (triệu VND)', fontsize=13)
axes[1].set_xlabel('Segment'); axes[1].set_ylabel('Triệu VND')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('chart4_cat_seg_revenue.png', dpi=150)
plt.show()

print('Top 3 category theo doanh thu:')
print(cat_rev.head(3).to_string())

## 4 - diagnostic analysis

In [ ]:
# chart 5: Seasonal Decompose Profit Margin 
ts = monthly.set_index('Date')['Profit_Margin'].asfreq('M')
result = seasonal_decompose(ts, model='additive', period=12)

fig = result.plot()
fig.set_size_inches(14, 10)
fig.suptitle('Phân tách Profit Margin: Trend + Seasonal + Residual', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('chart5_decompose.png', dpi=150)
plt.show()

# Xác định các tháng bất thường từ residual
residual = result.resid.dropna()
anomalies = residual[residual.abs() > residual.std() * 2].sort_values()
print('Các tháng bất thường trong residual (dùng cho Chart 6):')
print(anomalies.to_string())

# Lấy kỳ thật từ anomalies để phân tích tiếp
if len(anomalies):
    anom_years = anomalies.index.year.unique().tolist()
    problem_year_start = min(anom_years)
    problem_year_end = max(anom_years)
    print(f'\n=> Kỳ cần drill-down: {problem_year_start}-{problem_year_end}')
else:
    problem_year_start, problem_year_end = 2019, 2020
    print('Không có anomaly rõ ràng - dùng mặc định 2019-2020')

In [ ]:
# chart 6: Rev & COGS growth rate trong kỳ bất thường 
# Dùng đúng kỳ tìm được từ decompose ở trên (không hardcode)
problem_period = monthly[
    (monthly['year'] >= problem_year_start) &
    (monthly['year'] <= problem_year_end)
].copy()

problem_period['Rev_growth'] = problem_period['Revenue'].pct_change() * 100
problem_period['COGS_growth'] = problem_period['COGS'].pct_change() * 100

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(problem_period))
width = 0.35
ax.bar([i - width/2 for i in x], problem_period['Rev_growth'],
       width, label='Revenue growth %', color='steelblue')
ax.bar([i + width/2 for i in x], problem_period['COGS_growth'],
       width, label='COGS growth %',    color='tomato')
ax.set_xticks(list(x))
ax.set_xticklabels(problem_period['Date'].dt.strftime('%m/%Y'), rotation=45)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'Tốc độ tăng trưởng Revenue vs COGS ({problem_year_start}-{problem_year_end})', fontsize=13)
ax.set_ylabel('%'); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('chart6_rev_vs_cogs_growth.png', dpi=150)
plt.show()

In [ ]:
# chart 7: Sessions vs Conversion Rate
daily_orders = orders.groupby('order_date').size().reset_index(name='num_orders')
traffic_orders = traffic.merge(daily_orders, left_on='date', right_on='order_date', how='left')
traffic_orders['num_orders'] = traffic_orders['num_orders'].fillna(0)
traffic_orders['CR%'] = traffic_orders['num_orders'] / traffic_orders['sessions'] * 100
monthly_cr = traffic_orders.resample('M', on='date')[['sessions','num_orders','CR%']].mean()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(monthly_cr.index, monthly_cr['sessions'], alpha=0.4, label='Sessions', color='steelblue')
ax2.plot(monthly_cr.index, monthly_cr['CR%'], color='red', linewidth=2, label='CR%')
ax1.set_title('Sessions vs Conversion Rate (%) theo tháng', fontsize=13)
ax1.set_ylabel('Sessions'); ax2.set_ylabel('CR%')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('chart7_cr.png', dpi=150)
plt.show()

#cr% trung bình theo traffic_source
cr_by_source = traffic_orders.groupby(traffic_orders['traffic_source'])['CR%'].mean().sort_values(ascending=False)
print('CR% trung bình theo traffic source:')
print(cr_by_source.to_string())

In [ ]:
#chart 8: Return Rate theo Category
ret_prod = returns.merge(products[['product_id','category','size']], on='product_id')
items_cat = items_prod.groupby('category').size().reset_index(name='total_items')
ret_by_cat = ret_prod.groupby('category').size().reset_index(name='returns')
rate_cat = ret_by_cat.merge(items_cat, on='category')
rate_cat['return_rate%'] = rate_cat['returns'] / rate_cat['total_items'] * 100
rate_cat = rate_cat.sort_values('return_rate%', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(rate_cat['category'], rate_cat['return_rate%'], color='tomato')
mean_rr = rate_cat['return_rate%'].mean()
ax.axvline(mean_rr, color='navy', linestyle='--', label=f'Trung bình: {mean_rr:.1f}%')
ax.set_title('Return Rate (%) theo Category', fontsize=13)
ax.set_xlabel('Return Rate (%)')
ax.legend(); plt.tight_layout()
plt.savefig('chart8_return_rate.png', dpi=150)
plt.show()

top_cat = rate_cat.iloc[-1]['category']
reason = ret_prod[ret_prod['category'] == top_cat]['return_reason'].value_counts()
print(f'Lý do trả hàng cho {top_cat}:')
print(reason.to_string())

In [ ]:
# chart 9: Doanh thu theo Region
orders_geo = orders.merge(geo[['zip','region']], on='zip', how='left')
orders_geo_items = orders_geo.merge(
    items_prod[['order_id','revenue_line']].groupby('order_id').sum().reset_index(),
    on='order_id', how='left'
)
rev_by_region = orders_geo_items.groupby('region')['revenue_line'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(rev_by_region.index, rev_by_region.values / 1e6, color=['#2ecc71','#3498db','#e74c3c'])
ax.bar_label(bars, labels=[f'{v/1e6:.1f}M' for v in rev_by_region.values], padding=3)
ax.set_title('Tổng Doanh thu theo Region (triệu VND)', fontsize=13)
ax.set_xlabel('Region'); ax.set_ylabel('Triệu VND')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('chart9_region_revenue.png', dpi=150)
plt.show()

print(rev_by_region.to_string())

In [ ]:
#chart 10: Hiệu quả khuyến mãi
promo_orders = items[items['promo_id'].notna()].merge(
    orders[['order_id','order_date']], on='order_id'
).merge(promos[['promo_id','promo_name','promo_type']], on='promo_id')

promo_orders['revenue_line'] = (
    promo_orders['quantity'] * promo_orders['unit_price'] - promo_orders['discount_amount']
)

top_promos = (promo_orders.groupby('promo_name')['revenue_line']
              .sum().sort_values(ascending=False).head(10))

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_promos.index[::-1], top_promos.values[::-1] / 1e6, color='mediumpurple')
ax.set_title('Top 10 Chiến dịch Khuyến mãi theo Doanh thu (triệu VND)', fontsize=13)
ax.set_xlabel('Triệu VND')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('chart10_promo_effectiveness.png', dpi=150)
plt.show()

# Tỷ lệ đơn có promo vs không
promo_pct = items['promo_id'].notna().mean() * 100
print(f'Tỷ lệ order_items có khuyến mãi: {promo_pct:.1f}%')

In [ ]:
# chart 11: Customer - Inter-order Gap & Age Group

# Inter-order gap
cust_orders = orders.sort_values(['customer_id','order_date'])
cust_orders['prev_date'] = cust_orders.groupby('customer_id')['order_date'].shift(1)
cust_orders['gap_days']  = (cust_orders['order_date'] - cust_orders['prev_date']).dt.days
repeat_buyers = cust_orders.dropna(subset=['gap_days'])
median_gap = repeat_buyers['gap_days'].median()

# Orders per age_group
age_orders = (
    orders.merge(customers[['customer_id','age_group']], on='customer_id')
    .dropna(subset=['age_group'])
)
orders_per_age = (
    age_orders.groupby('age_group')['order_id'].count() /
    age_orders.groupby('age_group')['customer_id'].nunique()
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(repeat_buyers['gap_days'].clip(upper=365), bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(median_gap, color='red', linestyle='--', label=f'Median: {median_gap:.0f} ngày')
axes[0].set_title('Phân phối Inter-Order Gap (ngày)', fontsize=12)
axes[0].set_xlabel('Ngày giữa 2 lần mua')
axes[0].set_ylabel('Tần suất')
axes[0].legend()

axes[1].bar(orders_per_age.index, orders_per_age.values, color='teal')
axes[1].set_title('Số đơn TB / Khách hàng theo Age Group', fontsize=12)
axes[1].set_xlabel('Age Group'); axes[1].set_ylabel('Số đơn TB')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Hành vi mua hàng theo Khách hàng', fontsize=13)
plt.tight_layout()
plt.savefig('chart11_customer_behavior.png', dpi=150)
plt.show()

print(f'Median inter-order gap (repeat buyers): {median_gap:.0f} ngày')
print('\nSố đơn TB / KH theo age group:')
print(orders_per_age.to_string())

## 5 - predictive analysis

In [ ]:
# chart 12: YoY Revenue growth trend
yearly = sales.resample('Y', on='Date').sum().reset_index()
yearly['YoY_growth%'] = yearly['Revenue'].pct_change() * 100

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.bar(yearly['Date'].dt.year, yearly['Revenue']/1e6, color='steelblue', alpha=0.6, label='Revenue (triệu VND)')
ax2.plot(yearly['Date'].dt.year, yearly['YoY_growth%'], 'o-', color='red', linewidth=2, label='YoY Growth %')
ax2.axhline(0, color='black', linewidth=0.8)
ax1.set_title('Doanh thu theo năm & Tăng trưởng YoY (%)', fontsize=13)
ax1.set_xlabel('Năm'); ax1.set_ylabel('Triệu VND'); ax2.set_ylabel('YoY Growth %')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('chart12_yoy_growth.png', dpi=150)
plt.show()

print(yearly[['Date','Revenue','YoY_growth%']].to_string())

In [ ]:
# chart 13: Seasonal pattern trung bình theo tháng
seasonal_avg = monthly.groupby('month')['Revenue'].mean()
seasonal_idx = seasonal_avg / seasonal_avg.mean() * 100  # index 100 = average

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(1,13), seasonal_idx.values, color=[
    'tomato' if v < 90 else 'steelblue' for v in seasonal_idx.values
])
ax.axhline(100, color='black', linestyle='--', linewidth=1, label='Mức trung bình = 100')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12'])
ax.set_title('Chỉ số mùa vụ Doanh thu theo tháng (TB = 100)', fontsize=13)
ax.set_ylabel('Seasonal Index')
ax.bar_label(bars, labels=[f'{v:.0f}' for v in seasonal_idx.values], padding=2)
ax.legend(); plt.tight_layout()
plt.savefig('chart13_seasonal_index.png', dpi=150)
plt.show()

peak_month   = seasonal_idx.idxmax()
trough_month = seasonal_idx.idxmin()
print(f'Tháng đỉnh: T{peak_month} (index={seasonal_idx[peak_month]:.1f})')
print(f'Tháng đáy: T{trough_month} (index={seasonal_idx[trough_month]:.1f})')

In [ ]:
# chart 14: Inventory risk - stockout vs overstock
inv_risk = inventory.groupby('category').agg(
    stockout_rate = ('stockout_flag','mean'),
    overstock_rate= ('overstock_flag','mean'),
    avg_fill_rate = ('fill_rate','mean')
).reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(inv_risk))
w = 0.3
ax.bar([i-w for i in x], inv_risk['stockout_rate']*100,  w, label='Stockout rate %',  color='tomato')
ax.bar([i for i in x], inv_risk['overstock_rate']*100, w, label='Overstock rate %', color='steelblue')
ax.bar([i+w for i in x], inv_risk['avg_fill_rate']*100,  w, label='Fill rate %',      color='#2ecc71')
ax.set_xticks(list(x))
ax.set_xticklabels(inv_risk['category'], rotation=30)
ax.set_title('Rủi ro Tồn kho theo Category (%)', fontsize=13)
ax.set_ylabel('%'); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('chart14_inventory_risk.png', dpi=150)
plt.show()

high_stockout = inv_risk.nlargest(2, 'stockout_rate')[['category','stockout_rate']]
print('Category có stockout rate cao nhất:')
print(high_stockout.to_string())

## 6 - Prescriptive Analysis

In [ ]:
# prescriptive 1: Tiết kiệm từ giảm Return Rate
items_for_ref = items.merge(products[['product_id','category','cogs']], on='product_id')
items_by_cat2 = items_for_ref.groupby('category').size().reset_index(name='total_items')

ret_by_cat2 = ret_prod.groupby('category').size().reset_index(name='returns')
rate_cat2 = ret_by_cat2.merge(items_by_cat2, on='category')
rate_cat2['return_rate'] = rate_cat2['returns'] / rate_cat2['total_items']

avg_refund = returns['refund_amount'].mean()
target_rate = 0.10
years_data  = (sales.Date.max() - sales.Date.min()).days / 365

print('PRESCRIPTIVE: Tiết kiệm ước tính nếu giảm Return Rate về 10%')
print(f'Refund trung bình mỗi đơn trả: {avg_refund:,.0f} VND')
print(f'Dữ liệu span: {years_data:.1f} năm\n')

savings = []
for _, row in rate_cat2.iterrows():
    if row['return_rate'] > target_rate:
        annual_items  = row['total_items'] / years_data
        saving_annual = (row['return_rate'] - target_rate) * annual_items * avg_refund
        savings.append({
            'category':    row['category'],
            'current_rr':  row['return_rate']*100,
            'saving_VND':  saving_annual
        })

savings_df = pd.DataFrame(savings).sort_values('saving_VND', ascending=False)
print(savings_df.to_string(index=False))
print(f'\nTổng tiết kiệm ước tính/năm: {savings_df.saving_VND.sum():,.0f} VND')

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(savings_df['category'], savings_df['saving_VND']/1e6, color='#2ecc71')
ax.set_title('Tiết kiệm ước tính (triệu VND/năm) nếu giảm Return Rate => 10%', fontsize=12)
ax.set_xlabel('Triệu VND'); ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('chart15_prescriptive_return.png', dpi=150)
plt.show()

In [ ]:
#prescriptive 2: Tối ưu lịch khuyến mãi theo mùa vụ
# em nghĩ là: nên đẩy mạnh promo ở tháng đáy để nâng sàn doanh thu
trough_months = seasonal_idx[seasonal_idx < 90].index.tolist()
peak_months   = seasonal_idx[seasonal_idx > 110].index.tolist()

# Margin TB của promo vs non-promo orders
items_margin = items.copy()
items_margin['has_promo'] = items_margin['promo_id'].notna()
items_margin = items_margin.merge(products[['product_id','price','cogs']], on='product_id')
items_margin['gross_margin'] = (
    (items_margin['unit_price'] - items_margin['discount_amount']/items_margin['quantity'] - items_margin['cogs'])
    / items_margin['unit_price'] * 100
)

margin_comp = items_margin.groupby('has_promo')['gross_margin'].mean()
print('PRESCRIPTIVE: Margin khi có vs không có khuyến mãi')
print(f'Không promo: {margin_comp[False]:.1f}%')
print(f'Có promo: {margin_comp[True]:.1f}%')
print(f'Chênh lệch: {margin_comp[False]-margin_comp[True]:.1f} pp')

print(f'\n=> Tháng có seasonal index thấp (<90): {trough_months}')
print(f'=> Gợi ý: Tập trung promo vào tháng {trough_months} để nâng sàn,')
print(f'hạn chế discount ở tháng peak {peak_months} để bảo vệ margin')

In [ ]:
#prescriptive 3: Tối ưu tồn kho - định lượng cơ hội
#hmm maybe revenue bị mất vì stockout
stockout_months = inventory[inventory['stockout_flag'] == 1]
avg_daily_rev_per_product = (
    items_prod.merge(orders[['order_id','order_date']], on='order_id')
    .groupby('product_id')['revenue_line'].sum()
    / ((sales.Date.max() - sales.Date.min()).days)
)

stockout_merged = stockout_months.merge(
    avg_daily_rev_per_product.reset_index().rename(columns={'revenue_line':'daily_rev'}),
    on='product_id', how='left'
)
stockout_merged['lost_rev_est'] = stockout_merged['stockout_days'] * stockout_merged['daily_rev'].fillna(0)

lost_by_cat = stockout_merged.groupby('category')['lost_rev_est'].sum().sort_values(ascending=False)
print('PRESCRIPTIVE: Doanh thu ước tính bị mất do stockout'  )
print(lost_by_cat.apply(lambda x: f'{x/1e6:.2f}M VND').to_string())
print(f'\nTổng lost revenue ước tính: {lost_by_cat.sum()/1e6:.1f} triệu VND')

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(lost_by_cat.index[::-1], lost_by_cat.values[::-1]/1e6, color='tomato')
ax.set_title('Doanh thu ước tính bị mất do Stockout theo Category (triệu VND)', fontsize=12)
ax.set_xlabel('Triệu VND'); ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('chart16_prescriptive_stockout.png', dpi=150)
plt.show()